# 01 - Random Forest (binary: Normal vs Attack) on UNSW-NB15

Runs top to bottom in Colab (CPU runtime is fine). It will:
1. load the original UNSW-NB15 train/test CSVs (you upload them from your computer; CSV or Parquet both work),
2. preprocess them (encoder fit on **train only**),
3. train a Random Forest with class weighting,
4. report precision/recall, false-alarm rate, threshold trade-offs and per-attack-type detection,
5. save everything the backend needs and download a zip.

**Before you start:** download the UNSW-NB15 **training** and **testing** files (CSV or Parquet) from kaggle.com/datasets/mrwellsdavid/unsw-nb15 (Data tab).

In [ ]:
import os, json, glob, time, shutil, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, f1_score,
                             accuracy_score)
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
ART_DIR = os.environ.get("ART_DIR", "/content/artifacts")
os.makedirs(ART_DIR, exist_ok=True)
print("scikit-learn", sklearn.__version__)

In [ ]:
# Upload your two data files (train + test). CSV or Parquet both work.
from google.colab import files
os.makedirs("/content/data", exist_ok=True)

def data_files():
    found = []
    for ext in ("csv", "parquet"):
        found += glob.glob(f"/content/data/**/*.{ext}", recursive=True)
    return found

if len(data_files()) < 2:
    print("A file picker will appear: select BOTH the training and the testing file at once.")
    up = files.upload()
    for name in up:
        shutil.move(name, f"/content/data/{name}")
print("Files in /content/data:", [os.path.basename(p) for p in data_files()])

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "/content/data")

def find_split(keyword):
    hits = []
    for ext in ("parquet", "csv"):
        hits += glob.glob(os.path.join(DATA_DIR, "**", f"*{keyword}*.{ext}"), recursive=True)
    hits = sorted(set(hits))
    if not hits:
        found = [os.path.relpath(p, DATA_DIR)
                 for p in glob.glob(os.path.join(DATA_DIR, "**", "*.*"), recursive=True)]
        raise FileNotFoundError(
            f"No csv/parquet file with '{keyword}' in its name under {DATA_DIR}. Files found: {found}")
    if len(hits) > 1:
        print(f"Note: several files match '{keyword}', using the first:", [os.path.basename(h) for h in hits])
    return hits[0]

def read_any(path):
    df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

train_path, test_path = find_split("train"), find_split("test")
print("train:", train_path)
print("test :", test_path)
train_df, test_df = read_any(train_path), read_any(test_path)
print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)

for need in ("label", "attack_cat"):
    for name, df in (("train", train_df), ("test", test_df)):
        if need not in df.columns:
            raise KeyError(f"'{need}' column is missing in the {name} file. Columns are: {list(df.columns)}")

# If attack_cat is blank for normal flows, call them "Normal"
for df in (train_df, test_df):
    df["attack_cat"] = df["attack_cat"].astype(object)
    blank = df["attack_cat"].isna() & (df["label"].astype(int) == 0)
    df.loc[blank, "attack_cat"] = "Normal"

print("\nTrain label balance (0=normal, 1=attack):")
print(train_df["label"].astype(int).value_counts(normalize=True).round(3).to_string())
print("\nTrain attack_cat counts:")
print(train_df["attack_cat"].astype(str).str.strip().value_counts().to_string())

## Preprocess

In [ ]:
TARGET_COLS = ["label", "attack_cat"]

# Columns that identify a flow rather than describe its behaviour (a model could memorise them)
DROP_COLS = [c for c in ["id", "srcip", "dstip", "stime", "ltime"] if c in train_df.columns]
print("Dropping non-feature columns:", DROP_COLS)
for df in (train_df, test_df):
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
    df["attack_cat"] = df["attack_cat"].astype(str).str.strip()

if set(train_df.columns) != set(test_df.columns):
    raise ValueError(f"train/test columns differ: {set(train_df.columns) ^ set(test_df.columns)}")
test_df = test_df[train_df.columns.tolist()]

FEATURE_ORDER = [c for c in train_df.columns if c not in TARGET_COLS]

# Text columns (proto, service, state in the standard files) are the categorical ones
CAT_COLS = [c for c in FEATURE_ORDER if not pd.api.types.is_numeric_dtype(train_df[c])]
print("Categorical columns:", CAT_COLS)
for df in (train_df, test_df):
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip()

print(len(FEATURE_ORDER), "features")
assert len(FEATURE_ORDER) >= 40, "expected at least 40 features"
assert len(CAT_COLS) >= 1, "expected at least one categorical column"

# Encoder is fit on TRAIN only. Categories never seen in train (at test time or
# when a user uploads a file later) become -1 instead of crashing.
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.float32)
encoder.fit(train_df[CAT_COLS])

def make_X(df, enc=None):
    enc = enc or encoder
    X = df[FEATURE_ORDER].copy()
    X[CAT_COLS] = enc.transform(df[CAT_COLS])
    X = X.astype(np.float32)
    bad = int((~np.isfinite(X.values)).sum())
    if bad:
        print(f"  replacing {bad} NaN/inf values with 0")
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X

X_train, X_test = make_X(train_df), make_X(test_df)
unseen = int((X_test[CAT_COLS] == -1).any(axis=1).sum())
print(f"Test rows containing a category unseen in train: {unseen}")
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

In [ ]:
# Preprocessing artifacts the FastAPI backend will need (shared by both notebooks)
joblib.dump(encoder, f"{ART_DIR}/encoder.joblib")
with open(f"{ART_DIR}/feature_order.json", "w") as f:
    json.dump({
        "feature_order": FEATURE_ORDER,
        "categorical_cols": CAT_COLS,
        "categories": {c: [str(v) for v in cats] for c, cats in zip(CAT_COLS, encoder.categories_)},
    }, f, indent=2)

# A small demo CSV (up to 30 test flows per class, raw schema, labels kept) to upload in the GUI later
parts = [g.sample(min(len(g), 30), random_state=SEED) for _, g in test_df.groupby("attack_cat")]
sample = pd.concat(parts).sample(frac=1, random_state=SEED)
sample.to_csv(f"{ART_DIR}/sample_flows.csv", index=False)
print("Saved encoder.joblib, feature_order.json, sample_flows.csv (", len(sample), "rows )")

## Train the Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y_train = train_df["label"].astype(int).values
y_test = test_df["label"].astype(int).values

# Depth and leaf size are capped so the saved model stays a manageable size
# (GitHub rejects files over 100 MB).
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=25,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=SEED,
)
t0 = time.time()
rf.fit(X_train, y_train)
train_secs = time.time() - t0
print(f"Trained in {train_secs:.1f}s")

## Evaluate on the held-out test set

In [ ]:
t0 = time.time()
proba = rf.predict_proba(X_test)[:, 1]
infer_secs = time.time() - t0
infer_per_1000 = infer_secs / len(X_test) * 1000
pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, pred, target_names=["Normal", "Attack"], digits=4))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
fpr = fp / (fp + tn)   # normal flows wrongly flagged
fnr = fn / (fn + tp)   # attacks missed
roc = roc_auc_score(y_test, proba)
pr = average_precision_score(y_test, proba)
print(f"ROC-AUC {roc:.4f} | PR-AUC {pr:.4f} | false-alarm rate {fpr:.4f} | miss rate {fnr:.4f}")
print(f"Inference: {infer_per_1000:.4f} s per 1,000 flows")

print("\nThreshold trade-off (higher threshold = fewer false alarms, more missed attacks)")
for th in (0.3, 0.5, 0.7, 0.9):
    p = (proba >= th).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, p).ravel()
    print(f"  th={th:.1f}  precision={tp_/max(tp_+fp_,1):.3f}  recall={tp_/max(tp_+fn_,1):.3f}  false-alarm-rate={fp_/max(fp_+tn_,1):.3f}")

res = pd.DataFrame({"attack_cat": test_df["attack_cat"].values, "flagged": pred})
by_cat = res.groupby("attack_cat")["flagged"].agg(["mean", "count"])
by_cat.columns = ["share_flagged_as_attack", "n_flows"]
print("\nShare of flows flagged as attack, by true category")
print("(for Normal this is the false-alarm rate; for attacks it is the detection rate)")
print(by_cat.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
axes[0].set_title("Random Forest - confusion matrix")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

imp = pd.Series(rf.feature_importances_, index=FEATURE_ORDER).nlargest(15)
imp.sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 15 features (Gini importance)")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/results_rf.png", dpi=150)
plt.show()

## Save model + metrics, then verify the saved model reloads correctly

In [ ]:
model_path = f"{ART_DIR}/rf_binary.joblib"
joblib.dump(rf, model_path, compress=3)
size_mb = os.path.getsize(model_path) / 1e6
print(f"Saved {model_path} ({size_mb:.1f} MB)")
if size_mb > 90:
    print("WARNING: over ~90 MB. Don't commit this to GitHub; we'll host it elsewhere (or reduce n_estimators / max_depth).")

metrics = {
    "model": "RandomForestClassifier (binary)",
    "n_estimators": 200, "max_depth": 25, "min_samples_leaf": 2,
    "accuracy": accuracy_score(y_test, pred),
    "precision_attack": tp / max(tp + fp, 1), "recall_attack": tp / max(tp + fn, 1),
    "f1_attack": f1_score(y_test, pred),
    "roc_auc": roc, "pr_auc": pr,
    "false_alarm_rate": fpr, "miss_rate": fnr,
    "train_seconds": train_secs, "inference_seconds_per_1000_flows": infer_per_1000,
    "by_attack_type": by_cat.reset_index().to_dict(orient="records"),
}
with open(f"{ART_DIR}/metrics_rf.json", "w") as f:
    json.dump(metrics, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

versions_path = f"{ART_DIR}/versions.json"
versions = json.load(open(versions_path)) if os.path.exists(versions_path) else {}
versions.update({"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__})
with open(versions_path, "w") as f:
    json.dump(versions, f, indent=2)

# Sanity check: reload from disk and confirm identical predictions
rf2 = joblib.load(model_path)
enc2 = joblib.load(f"{ART_DIR}/encoder.joblib")
chunk = test_df.sample(1000, random_state=SEED)
Xc = make_X(chunk, enc2)
assert np.allclose(rf2.predict_proba(Xc)[:, 1], rf.predict_proba(Xc)[:, 1])
print("Reload check passed: saved model reproduces the same predictions.")

In [ ]:
shutil.make_archive("/content/rf_binary_artifacts", "zip", ART_DIR)
print("Created /content/rf_binary_artifacts.zip")

In [ ]:
from google.colab import files
files.download("/content/rf_binary_artifacts.zip")